In [ ]:
from gym_trading_env.downloader import download
import datetime
import time

# Timeframes to test
timeframes = ['1m']
symbols = ['BTC/USDT','ETH/USDT', 'ADA/USDT', 'XRP/USDT']  # You can add more like 'LTC/USDT' later

# Add delay between requests to avoid rate limits
DOWNLOAD_DELAY = 2  # seconds between downloads

print("🚀 DOWNLOADING COMPREHENSIVE DATASET FOR ANALYSIS")
print("=" * 60)

for timeframe in timeframes:
    try:
        filename = f'data/binance-BTCUSDT-{timeframe}.pkl'
        print(f"📥 Downloading {timeframe} data...")
        
        download(
            exchange_names=["binance"],
            symbols=symbols,
            timeframe=timeframe,
            dir="data",
            since=datetime.datetime(year=2010, month=2, day=1),
        )
        
        print(f"✅ Successfully downloaded {timeframe} data")
        
        # Add delay to avoid rate limiting
        if timeframe != timeframes[-1]:  # Don't delay after last download
            print(f"⏳ Waiting {DOWNLOAD_DELAY} seconds...")
            time.sleep(DOWNLOAD_DELAY)
            
    except Exception as e:
        print(f"❌ Failed to download {timeframe}: {e}")
        continue

print("=" * 60)
print("🎉 DOWNLOAD COMPLETE! Now run the analysis:")

🚀 DOWNLOADING COMPREHENSIVE DATASET FOR ANALYSIS
📥 Downloading 1m data...


In [ ]:


import pandas as pd


df = pd.read_pickle(filename)

print(f'Loaded {len(df)} ')

print(df.head(1))

In [ ]:
import pandas as pd
import numpy as np

timeframes = ['5m', '15m', '1h', '4h', '1d', '1w']

print("📊 COMPREHENSIVE TIMEFRAME ANALYSIS")
print("=" * 80)

results = []

for timeframe in timeframes:
    try:
        filename = f'data/binance-BTCUSDT-{timeframe}.pkl'
        df = pd.read_pickle(filename)
        returns = df['close'].pct_change().dropna()
        
        # Basic stats
        autocorr = returns.autocorr()
        mean_return = returns.mean()
        volatility = returns.std()
        positive_rate = (returns > 0).mean()
        sharpe = mean_return / volatility if volatility > 0 else 0
        
        # Data length and date range
        data_points = len(df)
        date_range = f"{df.index[0].strftime('%Y-%m-%d')} to {df.index[-1].strftime('%Y-%m-%d')}"
        
        # Store results
        results.append({
            'timeframe': timeframe,
            'autocorr': autocorr,
            'mean_return': mean_return,
            'volatility': volatility,
            'sharpe': sharpe,
            'positive_rate': positive_rate,
            'data_points': data_points,
            'date_range': date_range
        })
        
        print(f"{timeframe:4} | "
              f"Autocorr: {autocorr:7.4f} | "
              f"Mean: {mean_return:7.4%} | "
              f"Vol: {volatility:7.4%} | "
              f"Sharpe: {sharpe:7.4f} | "
              f"Positive: {positive_rate:6.1%} | "
              f"Samples: {data_points:6}")
              
    except Exception as e:
        print(f"{timeframe:4} | Error: {e}")

print("=" * 80)

# Find the best timeframe based on multiple criteria
if results:
    # Score each timeframe (higher = better)
    for result in results:
        score = 0
        # Prefer positive autocorrelation
        if result['autocorr'] > 0.02:
            score += result['autocorr'] * 10
        # Prefer reasonable volatility (1-5%)
        if 0.01 <= result['volatility'] <= 0.05:
            score += 2
        # Prefer positive mean returns
        if result['mean_return'] > 0:
            score += result['mean_return'] * 100
        # Prefer sufficient data points
        if result['data_points'] > 1000:
            score += 1
            
        result['score'] = score
    
    # Find best timeframe
    best = max(results, key=lambda x: x['score'])
    print(f"🎯 RECOMMENDED TIMEFRAME: {best['timeframe']} (Score: {best['score']:.2f})")
    print(f"   • Autocorrelation: {best['autocorr']:.4f}")
    print(f"   • Mean Return: {best['mean_return']:.4%}")
    print(f"   • Volatility: {best['volatility']:.4%}")
    print(f"   • Data Points: {best['data_points']}")
    print(f"   • Date Range: {best['date_range']}")

    